In [10]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

import string
import itertools
from collections import Counter

randseed_date = 20260623
np.random.seed(randseed_date)

## setup the 384 well format and the picklist of strains

In [12]:
pick_list = {'F17', 'O9', 'F5', 'G3', 'E24', 'C14', 'M19', 'E2', 'H11', 'E10', 'D23', 'E13', 'O11', 'G6', 'E4', 'A2', 'O13', 'I3', 'A12', 'G18', 'G4', 'O3', 'G9', 'A21', 'C22', 'E20', 'O21', 'A6', 'P15', 'A14', 'F15', 'G13', 'G12', 'P17', 'E15', 'M9', 'J5', 'E9', 'G1', 'J13', 'M17', 'E14', 'E7', 'E5', 'I9', 'J9', 'M11', 'H17', 'E6', 'P21', 'A1', 'G2', 'E12', 'O7', 'E16', 'A11', 'F23', 'P11', 'A15', 'E21', 'C24', 'O19', 'E22', 'G5', 'J1', 'A8', 'H23', 'G20', 'G8', 'C20', 'I11', 'G19', 'E8', 'G15', 'O17', 'G7', 'H15', 'G17', 'H19', 'E18', 'F19', 'M21', 'F7', 'G21', 'O15', 'M13', 'P19', 'G22', 'G11', 'F13'}
no_growth_strains = {'P23', 'H5', 'F21', 'G23', 'A10', 'J3', 'H7', 'J11', 'G10', 'G14'}
#pick list from plate reader data in the strain_check folder

n_rows, n_cols = 16, 24
wells384 = np.array([r + str(c) for r in string.ascii_uppercase[:n_rows] for c in range(1, n_cols + 1)])
wells384_2d = wells384.reshape(n_rows, n_cols)
# print(wells384)

# Step 1 — PCR primer layout (frame conditions)

Max out the PCR primer layout. There are **96 forward** and **96 reverse** barcodes. The only invalid combinations are where the forward and reverse barcode have the **same index** (`i == j`). Every other pairing is valid, giving `96 * 96 - 96 = 9120` combinations.

We lay each combination out into 384-well destination plates (`dest_row`, `dest_col`, `dest_plate`), randomize the placement with `np.random`, and emit the corresponding Echo transfer dataframe + CSV. `9120 / 384 = 23.75`, so this fills **24 plates** (the last one ~3/4 full).

In [2]:
# --- Source plate primer layout (same checkerboard scheme as the drug experiment) ---
n_rows_nonedge, n_cols_nonedge = n_rows - 2, n_cols - 2  # 14x22 = 308 wells for destination plates
fprimers_full = wells384_2d[::2, ::2].flatten()    # 96 forward primer source wells (even rows/cols)
rprimers_full = wells384_2d[1::2, 1::2].flatten()  # 96 reverse primer source wells (odd rows/cols)
n_fwd, n_rev = len(fprimers_full), len(rprimers_full)
print(f"forward primers: {n_fwd}, reverse primers: {n_rev}")

# --- every valid forward x reverse combination (exclude identical index pairs) ---
combos = []
for i in range(n_fwd):
    for j in range(n_rev):
        if i != j:
            combos.append((i, j))
combos = np.array(combos)
print(f"total valid combos: {len(combos)} (expected {n_fwd * n_rev - n_fwd})")

# randomize plate placement
np.random.shuffle(combos)

# --- assign each combo to a destination row / col / plate, filling plates row-major ---

wells_per_plate = n_rows_nonedge * n_cols_nonedge  # 308
n_plates = int(np.ceil(len(combos) / wells_per_plate))
print(f"plates needed: {n_plates}")

records = []
for idx, (fi, rj) in enumerate(combos):
    plate  = idx // wells_per_plate + 1
    within = idx %  wells_per_plate 
    row_i  = (within // n_cols_nonedge) + 1  # skip first row (A)
    col_i  = (within %  n_cols_nonedge  ) + 1  # skip first col (1)
    dest_row = string.ascii_uppercase[row_i]
    dest_col = col_i + 1
    records.append({
        'dest_plate': plate,
        'dest_row': dest_row,
        'dest_col': dest_col,
        'dest_well': f"{dest_row}{dest_col}",
        'fwd_idx': int(fi),
        'rev_idx': int(rj),
        'fwd_source_well': fprimers_full[fi],
        'rev_source_well': rprimers_full[rj],
    })
df_layout = pd.DataFrame(records)

# --- sanity checks ---
assert (df_layout['fwd_idx'] == df_layout['rev_idx']).sum() == 0, "found identical-index pairs"
assert df_layout[['fwd_idx', 'rev_idx']].drop_duplicates().shape[0] == len(df_layout), "duplicate combos"
assert set(df_layout['fwd_source_well']).isdisjoint(set(df_layout['rev_source_well'])), "fwd/rev source collision"
print(f"last plate fill: {(df_layout['dest_plate'] == n_plates).sum()} / {wells_per_plate}")
display(df_layout)

forward primers: 96, reverse primers: 96
total valid combos: 9120 (expected 9120)
plates needed: 30
last plate fill: 188 / 308


,dest_plate,dest_row,dest_col,dest_well,fwd_idx,rev_idx,fwd_source_well,rev_source_well
0,1,B,2,B2,7,15,A15,D8
1,1,B,3,B3,3,80,A7,N18
2,1,B,4,B4,69,12,K19,D2
3,1,B,5,B5,85,61,O3,L4
4,1,B,6,B6,50,41,I5,H12
...,...,...,...,...,...,...,...,...
9115,30,J,9,J9,48,26,I1,F6
9116,30,J,10,J10,23,83,C23,N24
9117,30,J,11,J11,15,2,C7,B6
9118,30,J,12,J12,9,39,A19,H8


In [ ]:
# --- Echo transfer df: two rows per destination well (forward primer + reverse primer) ---
vol = 500  # nL per primer transfer (matches the drug experiment)

echo_rows = []
for r in df_layout.itertuples():
    echo_rows.append({'Source Well': r.fwd_source_well, 'Destination Plate Name': r.dest_plate,
                      'Destination Well': r.dest_well, 'Transfer Volume': vol})
    echo_rows.append({'Source Well': r.rev_source_well, 'Destination Plate Name': r.dest_plate,
                      'Destination Well': r.dest_well, 'Transfer Volume': vol})
df_echo = pd.DataFrame(echo_rows)
print(f"echo rows: {len(df_echo)} (2 x {len(df_layout)} = {2 * len(df_layout)})")

# full layout (with primer indices/source wells) and the Echo-ready transfer file
df_layout.to_csv(f"primer_layout_{randseed_date}.csv", index=False)
df_echo.to_csv(f"echo_primers_{randseed_date}.csv", index=False)
#split df_echo into two files to give a break for refilling
df_echo_A = df_echo.iloc[:len(df_echo)//2]
df_echo_B = df_echo.iloc[len(df_echo)//2:]
df_echo_A.to_csv(f"echo_primers_A_{randseed_date}.csv", index=False)
df_echo_B.to_csv(f"echo_primers_B_{randseed_date}.csv", index=False)

display(df_echo_A)
display(df_echo_B)

echo rows: 18240 (2 x 9120 = 18240)


,Source Well,Destination Plate,Destination Well,Transfer Volume
0,A15,1,B2,500
1,D8,1,B2,500
2,A7,1,B3,500
3,N18,1,B3,500
4,K19,1,B4,500
...,...,...,...,...
9115,D6,15,M5,500
9116,C9,15,M6,500
9117,H2,15,M6,500
9118,K23,15,M7,500


,Source Well,Destination Plate,Destination Well,Transfer Volume
9120,O5,15,M8,500
9121,D18,15,M8,500
9122,E15,15,M9,500
9123,F20,15,M9,500
9124,E23,15,M10,500
...,...,...,...,...
18235,B6,30,J11,500
18236,A19,30,J12,500
18237,H8,30,J12,500
18238,K23,30,J13,500


In [ ]:
#sum how much will be shot from each well with a groupby on df_strain_echo
primer_shots = df_echo.groupby('Source Well')['Transfer Volume'].sum()
display(primer_shots)

# Step 2 — assign strain pairs + build the strain Echo script

For every destination plate/well from Step 1 we assign a random **pair of strains** (the pairwise-interaction condition) or, for controls, a single strain by itself.

**Source plate:** a 384-well strain layout.

**Constraints (90 strains → >3741 unordered pairs, 9120 wells over 30 plates):**
- Every pair gets **≤ 3** replicates.
- Plates **1–19** are filled with a balanced random pass (each pair 1–2×); plates **20–30** then top every under-covered pair up to **≥ 2** replicates and carry the extra 3rd reps — so "starting at plate 20, every pair has at least 2".
- **≥ 1 monoculture control** per strain (100 nL strain + 100 nL blank from source well `A1`), scattered across all 30 plates.

Resulting allocation: 2190 pairs ×2 + 1551 pairs ×3 + 87 monocultures = 9120 wells.

In [18]:
strainseed = randseed_date + 1
rng = np.random.default_rng(strainseed)

strains = list(pick_list)
assume_blank = list(no_growth_strains)
N = len(strains)

pairs = list(itertools.combinations(range(N), 2))   # unordered strain pairs (by index)
P = len(pairs)
total = len(df_layout)
print(f"{P} unordered pairs, {total} destination wells")

# content[i] = ('mono', s) or ('pair', s1, s2) for destination-well row i of df_layout
content = [None] * total

# --- 1) monoculture controls: 1 per strain, scattered across all plates ---
mono_wells = rng.choice(total, size=N, replace=False)
for w_i, s_i in zip(mono_wells, rng.permutation(N)):
    content[w_i] = ('mono', strains[s_i])
mono_set = set(mono_wells.tolist())

# --- 2) split the remaining wells into early (plates 1-19) and late (plates 20+) ---
plate_of = df_layout['dest_plate'].values
rem   = [i for i in range(total) if i not in mono_set]
early = [i for i in rem if plate_of[i] <= 19]
late  = [i for i in rem if plate_of[i] >= 20]
n_early, n_late = len(early), len(late)
assert n_early >= P and (n_early - P) <= P, "early region cannot hold a balanced 1-2x pass"

# phase 1 (plates 1-19): every pair 1x, then a random subset gets a 2nd rep to fill the region
reps = np.ones(P, dtype=int)
second = rng.choice(P, size=n_early - P, replace=False)
reps[second] += 1
early_inst = list(range(P)) + list(second)
rng.shuffle(early_inst)
for w_i, p in zip(early, early_inst):
    a, b = pairs[p]
    content[w_i] = ('pair', strains[a], strains[b])

# phase 2 (plates 20+): complete every under-covered pair to >=2, then add 3rd reps (cap 3)
cur = reps.copy()
completion = [p for p in range(P) if cur[p] < 2]   # pairs still at 1 rep
for p in completion:
    cur[p] += 1
n_third = n_late - len(completion)
elig3 = [p for p in range(P) if cur[p] == 2]
third = rng.choice(elig3, size=n_third, replace=False)
late_inst = completion + list(third)
rng.shuffle(late_inst)
for w_i, p in zip(late, late_inst):
    a, b = pairs[p]
    content[w_i] = ('pair', strains[a], strains[b])
print(f"plates 1-19: {n_early} pair wells | plates 20+: {len(completion)} min-2 completions + {n_third} third reps")

# --- write strain assignment back onto df_layout ---
assert all(c is not None for c in content), "unassigned wells remain"
df_layout['well_type'] = [c[0] for c in content]
df_layout['strain1']   = [c[1] for c in content]
df_layout['strain2']   = [c[2] if c[0] == 'pair' else np.nan for c in content]
display(df_layout[['dest_plate', 'dest_well', 'well_type', 'strain1', 'strain2']])

#fill the strain2 column of the mono rows with a random selection from no_growth_strains
blank_well = rng.choice(assume_blank)
df_layout.loc[df_layout['well_type'] == 'mono', 'strain2'] = rng.choice(assume_blank, size=(df_layout['well_type'] == 'mono').sum(), replace=True)
display(df_layout[df_layout['well_type'] == 'mono'])


4005 unordered pairs, 9120 destination wells
plates 1-19: 5796 pair wells | plates 20+: 2214 min-2 completions + 1020 third reps


,dest_plate,dest_well,well_type,strain1,strain2
0,1,B2,pair,E16,E15
1,1,B3,pair,E9,G11
2,1,B4,pair,E9,H19
3,1,B5,pair,P11,E14
4,1,B6,pair,F7,F17
...,...,...,...,...,...
9115,30,J9,pair,E20,F19
9116,30,J10,pair,C14,H11
9117,30,J11,pair,A1,G2
9118,30,J12,pair,E6,O9


,dest_plate,dest_row,dest_col,dest_well,fwd_idx,rev_idx,fwd_source_well,rev_source_well,well_type,strain1,strain2
178,1,J,4,J4,82,12,M21,D2,mono,E24,G23
199,1,K,3,K3,90,20,O13,D18,mono,O3,F21
227,1,L,9,L9,22,81,C21,N20,mono,E21,J11
306,1,O,22,O22,12,25,C1,F4,mono,M13,G23
360,2,D,10,D10,43,89,G15,P12,mono,G6,H7
...,...,...,...,...,...,...,...,...,...,...,...
8635,29,B,13,B13,39,8,G7,B18,mono,G7,P23
8716,29,F,6,F6,27,70,E7,L22,mono,M9,J11
8923,29,O,15,O15,57,78,I19,N14,mono,A12,H7
8929,29,O,21,O21,85,3,O3,B8,mono,H23,G10


In [ ]:
# --- Strain Echo transfer df: two 100 nL transfers per destination well ---
strain_vol = 100  # nL per strain transfer
# strain_vol_div = 4
strain_vol_div = strain_vol // 25  # 25 nL per transfer (4 transfers per strain)    
strain_vol_div = False

strain_rows = []
for r in df_layout.itertuples():
    if r.well_type == 'pair':
        src_a, src_b = r.strain1, r.strain2
    else:  # monoculture control: strain + blank/media
        #src_a, src_b = r.strain1, blank_well
        src_a, src_b = r.strain1, r.strain2
    for src in (src_a, src_b):
        if strain_vol_div:
            for i in range(strain_vol_div):  # 4 transfers per strain to reach 100 nL total
                strain_rows.append({'Source Well': src, 'Destination Plate Name': r.dest_plate,
                                'Destination Well': r.dest_well, 'Transfer Volume': 25})
        else:
            strain_rows.append({'Source Well': src, 'Destination Plate Name': r.dest_plate,
                                'Destination Well': r.dest_well, 'Transfer Volume': strain_vol})
df_strain_echo = pd.DataFrame(strain_rows)

# --- sanity checks ---
pair_reps = Counter()
mono_counts = Counter()
for c in content:
    if c[0] == 'pair':
        pair_reps[frozenset((c[1], c[2]))] += 1
    else:
        mono_counts[c[1]] += 1
rc = np.array(list(pair_reps.values()))
assert len(pair_reps) == P, "not every pair is present"
assert rc.min() >= 2 and rc.max() <= 3, "replicate bounds violated"
assert all(mono_counts[s] >= 1 for s in strains), "a strain is missing its monoculture control"
if strain_vol_div:
    assert len(df_strain_echo) == 8 * total
else:
    assert len(df_strain_echo) == 2 * total
print(f"pairs: {len(pair_reps)}/{P} | reps min={rc.min()} max={rc.max()} | "
      f"2x={(rc==2).sum()} 3x={(rc==3).sum()}")
print(f"monoculture controls: {len(mono_counts)} strains, all >=1 rep")
print(f"strain echo rows: {len(df_strain_echo)}")

# --- save Echo-ready files (full + A/B halves for a refill break, mirroring the primer step) ---
df_layout.to_csv(f"strain_layout_{randseed_date}.csv", index=False)
df_strain_echo.to_csv(f"echo_strains_{randseed_date}.csv", index=False)
half = len(df_strain_echo) // 2
df_strain_echo.iloc[:half].to_csv(f"echo_strains_A_{randseed_date}.csv", index=False)
df_strain_echo.iloc[half:].to_csv(f"echo_strains_B_{randseed_date}.csv", index=False)
display(df_strain_echo)

pairs: 4005/4005 | reps min=2 max=3 | 2x=2985 3x=1020
monoculture controls: 90 strains, all >=1 rep
strain echo rows: 18240


,Source Well,Destination Plate,Destination Well,Transfer Volume
0,E16,1,B2,100
1,E15,1,B2,100
2,E9,1,B3,100
3,G11,1,B3,100
4,E9,1,B4,100
...,...,...,...,...
18235,G2,30,J11,100
18236,E6,30,J12,100
18237,O9,30,J12,100
18238,A15,30,J13,100


In [22]:
#sum how much will be shot from each well with a groupby on df_strain_echo
strain_shots = df_strain_echo.groupby('Source Well')['Transfer Volume'].sum()
display(strain_shots)

Source Well
A1     19100
A10      800
A11    19800
A12    19600
A14    20400
       ...  
P15    20100
P17    19300
P19    19800
P21    20300
P23      900
Name: Transfer Volume, Length: 100, dtype: int64